# Document Q&A using Amazon Nova 2 Lite in SageMaker Studio

> *This notebook should work well with the **`Data Science 3.0`** kernel in SageMaker Studio*

---

## 🏗️ What This Lab Builds

This lab demonstrates a simple **Document Q&A pipeline** using **Amazon Nova 2 Lite** via Amazon Bedrock.

```
documents.txt  ──┐
                 ├──► Amazon Nova 2 Lite ──► Natural Language Answer
User Query     ──┘
```

Nova reads all the documents as context and answers your question directly in natural language.

---

## 🔄 What Changed vs the Original Lab?

| Item | Original Lab | This Lab |
|---|---|---|
| **Model** | `amazon.titan-embed-text-v1` | `us.amazon.nova-2-lite-v1:0` ✅ |
| **Embeddings** | ✅ Yes | ❌ Not needed |
| **Vector math** | ✅ Cosine similarity | ❌ Not needed |
| **Answer style** | Prints raw document line | Natural language answer ✅ |
| **Architecture** | Similarity search | Direct document Q&A ✅ |

> **⚠️ Prerequisite:** Ensure `us.amazon.nova-2-lite-v1:0` is accessible in Amazon Bedrock → Model catalog (us-east-1)

## Step 1 — Setup

Import libraries and create a Bedrock Runtime client.
We only need **one model** — Amazon Nova 2 Lite — for this entire lab.

In [ ]:
import json
import boto3

# Bedrock Runtime client
boto3_bedrock = boto3.client(service_name='bedrock-runtime')

# Amazon Nova 2 Lite — Active model (replaces deprecated nova-lite-v1:0)
modelId = "us.amazon.nova-2-lite-v1:0"

print(f"Model: {modelId}")
print("Setup complete!")

## Step 2 — Load Documents

Read all content from `documents.txt` — this becomes the **knowledge base** that Nova will use to answer questions.

Nova reads the entire document as context, so no embeddings or vector math is needed.

In [ ]:
# Load all documents from documents.txt into a single string
with open("documents.txt") as f:
    knowledge_base = f.read()

print("Knowledge Base Loaded:")
print("─" * 40)
print(knowledge_base)
print("─" * 40)

## Step 3 — Interactive Q&A Loop with Amazon Nova 2 Lite

Type any question — Nova will read the documents and answer in natural language.

### How it works:
1. Your question is sent to **Amazon Nova 2 Lite** along with all the documents as context
2. Nova reads the context and generates a natural language answer
3. If the answer is not in the documents, Nova replies **"I don't know"**
4. Type `quit` to exit

### Try these queries:
- `What is the price of product number 5723?` → Nova answers **$76** ✅
- `What is the price of iPhone 15?` → Nova says **I don't know** ✅
- `quit` → Exits the loop

In [ ]:
while True:
    query = input("\nEnter your query or say quit to exit: ")

    if query == "quit":
        print("Exiting. Goodbye!")
        break

    # Build the prompt — Nova receives the full knowledge base + user question
    user_prompt = f"""Use the following documents to answer the question.
If the answer is not present in the documents, say 'I don't know'.

Documents:
{knowledge_base}

Question: {query}

Answer:"""

    # Nova 2 Lite request body
    request_body = {
        "messages": [
            {
                "role": "user",
                "content": [{"text": user_prompt}]
            }
        ],
        "system": [
            {
                "text": (
                    "You are a helpful assistant. "
                    "Answer ONLY from the provided documents. "
                    "If the answer is not in the documents, say 'I don't know'."
                )
            }
        ],
        "inferenceConfig": {
            "maxTokens": 300,
            "temperature": 0.1
        }
    }

    try:
        response = boto3_bedrock.invoke_model(
            body=json.dumps(request_body),
            modelId=modelId,
            accept="application/json",
            contentType="application/json",
        )
        response_body = json.loads(response.get("body").read())
        answer = response_body["output"]["message"]["content"][0]["text"].strip()

        print("\nQuery: " + query)
        print("Answer: " + answer)

    except Exception as e:
        print(f"Error: {e}")